In [8]:
import json
import os
import numpy as np
import openai
import yaml
import zipfile
from functools import lru_cache
import uuid


In [9]:
tools_md = "data/tools_metadata.json"
with open(tools_md, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Anzahl der Tools: {len(data)}")

Anzahl der Tools: 3696


In [10]:
from openai import OpenAI

client = OpenAI(base_url="https://llm.scads.ai/v1",
                api_key=os.environ.get('SCADSAI_API_KEY'))

In [11]:
def embed(text):
    if not text or not text.strip():
        return None
    response = client.embeddings.create(
        input=text,
        model="Qwen/Qwen3-Embedding-4B"
    )
    return response.data[0].embedding

In [12]:
def build_doc(owner, name, t):
    parts = [
        t.get("name") or name,
        t.get("description") or "",
        t.get("repo_description") or "",
        t.get("repo_long_description") or "",
        t.get("detailed_description_generated") or ""
    ]
    # knappe, saubere Repräsentation
    text = " ".join(" ".join(parts).split())
    meta = {
        "owner": owner,
        "repo_name": name,
        "tool_id": t.get("tool_id"),
        "version": t.get("version"),
        "guid": t.get("guid"),
    }
    return text, meta

def build_all_docs(data):
    texts, metas = [], []
    for entry in data:
        owner, repo_name = entry["owner"], entry["name"]
        for t in entry.get("tools", []):
            txt, m = build_doc(owner, repo_name, t)
            if txt:
                texts.append(txt)
                metas.append(m)
    return texts, metas



In [13]:
texts, metas = build_all_docs(data)
print(f"Dokumente für Embedding: {len(texts)}")

Dokumente für Embedding: 3787


In [14]:
# 4) VectorStore anpassen: Metadaten mitführen
class VectorStore:
    def __init__(self, embed_fn, texts=None, metadatas=None,
                 dtype=np.float32, normalize=True):
        self.embed_fn = embed_fn
        self.dtype = dtype
        self.normalize = normalize
        self.vectors = np.empty((0, 0), dtype=dtype)  # (n_docs, dim)
        self.texts = []
        self.metadatas = []
        if texts:
            self.add(texts, metadatas)

    def _prep_vec(self, v):
        a = np.asarray(v, dtype=self.dtype)
        if self.normalize:
            n = np.linalg.norm(a)
            if n > 0:
                a = a / n
        return a

    def add(self, texts, metadatas=None):
        if isinstance(texts, str):
            texts = [texts]
        if metadatas is None:
            metadatas = [None] * len(texts)
        embs = []
        for t in texts:
            e = self.embed_fn(t)
            if e is None:
                # überspringe leere Texte
                continue
            embs.append(self._prep_vec(e))
        if not embs:
            return
        M = np.vstack(embs)
        if self.vectors.size == 0:
            self.vectors = M
        else:
            if M.shape[1] != self.vectors.shape[1]:
                raise ValueError("Alle Embeddings müssen dieselbe Dimension haben.")
            self.vectors = np.vstack([self.vectors, M])
        self.texts.extend(texts[:len(embs)])
        self.metadatas.extend(metadatas[:len(embs)])

    def search(self, query, k=3, return_scores=False):
        if len(self.texts) == 0:
            return [] if not return_scores else []
        q = self._prep_vec(self.embed_fn(query))
        if self.vectors.size == 0:
            return [] if not return_scores else []
        sims = self.vectors @ q
        k = min(k, len(self.texts))
        idx = np.argpartition(-sims, k - 1)[:k]
        idx = idx[np.argsort(-sims[idx])]
        results = [
            {"text": self.texts[i], "meta": self.metadatas[i]}
            for i in idx
        ]
        if return_scores:
            return [(results[j], float(sims[idx[j]])) for j in range(len(idx))]
        return results


In [15]:
# 5) Store korrekt instanziieren
vector_store = VectorStore(embed_fn=embed, texts=texts, metadatas=metas)

# # 6) Beispiel-Query
# hits = vector_store.search("Adapter aus FASTQ entfernen und PE-Reads zusammenführen", k=5)
# for h in hits:
#     print("----")
#     for key, value in h["meta"].items():
#         print(f"{key}: {value}")

KeyboardInterrupt: 

In [8]:
knwf_path = "/Users/khanh/Repository/imaging_KNIME_to_Galaxy/src/knime2galaxy/data/file_to_translate/2025_03_2D_spot_detection.knwf"


In [9]:
def collect_knime_node_files(knwf_path: str) -> dict:
    """
    Collects all node settings.xml files inside the KNIME .knwf archive.
    Returns a dictionary: {node_folder_name: xml_content}.
    """
    node_data = {}
    with zipfile.ZipFile(knwf_path, "r") as zf:
        for file_name in zf.namelist():
            if file_name.endswith("settings.xml"):
                with zf.open(file_name) as f:
                    xml_content = f.read().decode("utf-8")
                    node_name = file_name.split("/")[-2]  # Ordnername
                    node_data[node_name] = xml_content
    return node_data

In [10]:
knime_nodes = collect_knime_node_files(knwf_path)
print("First 100 characters in every node:")
for node_name, xml_content in knime_nodes.items():
    print(f"{node_name}: {xml_content[:100]}...")

First 100 characters in every node:
Labeling Filter (#8): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
Spot Detection (#4): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
CLAHE (#3): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
CSV Writer (#9): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
Image Reader (#1): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
Connected Component Analysis (#5): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
Image Segment Features (#6): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...


In [11]:
# Collects the workflow.knime file
def collect_workflow_file(knwf_path: str) -> str:
    """
    Extracts the content of the workflow.knime file inside the KNIME .knwf archive.
    Returns the file content as a string.
    """
    with zipfile.ZipFile(knwf_path, "r") as zf:
        for file_name in zf.namelist():
            if file_name.endswith("workflow.knime"):
                with zf.open(file_name) as f:
                    return f.read().decode("utf-8")
    raise FileNotFoundError("workflow.knime not found in KNWF archive")

In [12]:
# Output the first 500 characters of the workflow
workflow_content = collect_workflow_file(knwf_path)
print(workflow_content[:500])  

<?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="workflow.knime">
    <entry key="created_by" type="xstring" value="4.7.7.v202308161346"/>
    <entry key="created_by_nightly" type="xboolean" value="false"/>
    <entry key="version" type="xstring" value="4.1.0"/>
    <entry key="name" type="xs


In [13]:
@lru_cache(maxsize=1)
def load_translation_examples(yaml_path: str = "data/translation_table.yml") -> list:
    with open(yaml_path, "r", encoding="utf-8") as f:
        docs = list(yaml.safe_load_all(f))
        print(docs)
        examples = []
        for doc in docs[0]:
            knime = doc.get("KNIME")
            galaxy = doc.get("Galaxy")

            if knime and galaxy:
                examples.append({
                    "KNIME": knime.strip(),
                    "Galaxy": galaxy.strip()
                })

    return examples


In [14]:
translation_examples = load_translation_examples(yaml_path="data/translation_table.yml")
translation_examples

[[{'description': 'Input Image dataset', 'Galaxy': '"0": {\n        "annotation": "",\n        "content_id": null,\n        "errors": null,\n        "id": 0,\n        "input_connections": {},\n        "inputs": [\n            {\n                "description": "",\n                "name": "Input Image Dataset"\n            }\n        ],\n        "label": "Input Image Dataset",\n        "name": "Input dataset",\n        "outputs": [],\n        "position": {\n            "left": 0.0,\n            "top": 0.0\n        },\n        "tool_id": null,\n        "tool_state": "{\\"optional\\": false, \\"tag\\": null}",\n        "tool_version": null,\n        "type": "data_input",\n        "uuid": "312178b3-3fac-49ab-95cf-29e3f44e6015",\n        "when": null,\n        "workflow_outputs": []\n    },\n', 'Python': "from skimage import io\n\ninput_image = io.imread('input_image.tiff')\n", 'KNIME': '<?xml version="1.0" encoding="UTF-8"?>\n<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi

[{'KNIME': '<?xml version="1.0" encoding="UTF-8"?>\n<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="settings.xml">\n    <entry key="node_file" type="xstring" value="settings.xml"/>\n    <config key="flow_stack"/>\n    <config key="internal_node_subsettings">\n        <entry key="memory_policy" type="xstring" value="CacheSmallInMemory"/>\n    </config>\n    <config key="model">\n        <config key="check_file_format_Internals">\n            <entry key="SettingsModelID" type="xstring" value="SMID_boolean"/>\n            <entry key="EnabledStatus" type="xboolean" value="true"/>\n        </config>\n        <entry key="check_file_format" type="xboolean" value="true"/>\n        <config key="group_files_Internals">\n            <entry key="SettingsModelID" type="xstring" value="SMID_boolean"/>\n            <entry key="Enabled

In [15]:
def build_examples():
    examples_text =  """You are a translator that converts KNIME workflow nodes to Galaxy workflow steps.

Below are examples of how this translation should be done:
"""
    examples = load_translation_examples()
    if examples:
        for i, ex in enumerate(examples[:6]):  # z. B. nur 6 Beispiele
            examples_text += f"""

## Example {i + 1}:

KNIME node (XML):

```xml
{ex["KNIME"]}
```

Galaxy step (JSON):
```json
{ex["Galaxy"]}
```

"""
    return examples_text


In [16]:
node_examples = build_examples()
print(node_examples)

[[{'description': 'Input Image dataset', 'Galaxy': '"0": {\n        "annotation": "",\n        "content_id": null,\n        "errors": null,\n        "id": 0,\n        "input_connections": {},\n        "inputs": [\n            {\n                "description": "",\n                "name": "Input Image Dataset"\n            }\n        ],\n        "label": "Input Image Dataset",\n        "name": "Input dataset",\n        "outputs": [],\n        "position": {\n            "left": 0.0,\n            "top": 0.0\n        },\n        "tool_id": null,\n        "tool_state": "{\\"optional\\": false, \\"tag\\": null}",\n        "tool_version": null,\n        "type": "data_input",\n        "uuid": "312178b3-3fac-49ab-95cf-29e3f44e6015",\n        "when": null,\n        "workflow_outputs": []\n    },\n', 'Python': "from skimage import io\n\ninput_image = io.imread('input_image.tiff')\n", 'KNIME': '<?xml version="1.0" encoding="UTF-8"?>\n<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi

In [17]:
# Convert dict content to string.
knime_nodes_str = "\n".join(
    f"Node ID: {key}\n{value}" for key, value in knime_nodes.items()
)
print(knime_nodes_str)

Node ID: Labeling Filter (#8)
<?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="settings.xml">
    <entry key="node_file" type="xstring" value="settings.xml"/>
    <config key="flow_stack"/>
    <config key="internal_node_subsettings">
        <entry key="memory_policy" type="xstring" value="CacheSmallInMemory"/>
    </config>
    <config key="model">
        <config key="column_creation_mode_Internals">
            <entry key="SettingsModelID" type="xstring" value="SMID_string"/>
            <entry key="EnabledStatus" type="xboolean" value="true"/>
        </config>
        <entry key="column_creation_mode" type="xstring" value="Append"/>
        <config key="column_suffix_Internals">
            <entry key="SettingsModelID" type="xstring" value="SMID_string"/>
            <entry ke

In [18]:
def build_summary_prompt():
    
    summary_prompt = f"""
# Your Task
You are a rigorous workflow graph extractor and validator.
Your job is to read KNIME workflow XML and produce a clean structural summary of how nodes are connected,
so that the workflow can later be converted into a Galaxy (.ga) workflow.

You must also detect structural and semantic validation errors in the workflow (e.g. missing outputs, invalid connections, type mismatches).
You must not hallucinate any connections, nodes, or ports.
Respond only with valid JSON in the format below — no free text, no comments.

# Input
KNIME Nodes (XML):
```xml
{knime_nodes_str}
```

The KNIME workflow content (XML):
```xml
{workflow_content}
```

# Core Extraction Rules
- Extract every node (<node id="...">) with: id, label or name, kind (default: "op"), all explicit or implied input/output ports (from connections).
- Extract every data connection (<connection sourceID="..." sourcePort="..." destID="..." destPort="..."/>).
- Derive: "entry" = nodes with no incoming edges, "exit" = nodes with no outgoing edges
- Maintain stable node IDs — do not renumber.

# Validation Rules 
You must verify workflow consistency before outputting the final graph.
Report any violations in "validation_errors" with "severity", "rule", "message", and "evidence".
Input-type correctness

Nodes of type "data_input" do not produce outputs.

If another node references an "output" from a data_input, this is invalid → mark as "invalid_reference".

Expected fix: change the node type to "data_collection_input" or define an explicit output field.

Output-definition completeness

Any node that appears as a connection source must define at least one output.

If no output definition exists, mark as "missing_output_definition".

Connection validity

Every edge must reference an existing (node, port) pair.

If the referenced node or port does not exist, mark as "invalid_reference".

Type compatibility

Check that connected nodes have compatible data types (e.g. data_input → data_collection_input is invalid).

If the types differ in an incompatible way, mark as "type_mismatch".

Referential integrity

Each "output_name" in a connection must exist in the "outputs" list of the source node.

Each "input_name" must have a valid upstream output.

Also describe in words how the nodes are connected and what the workflow does. 

"""
    
    return summary_prompt

In [19]:
summary_task = build_summary_prompt()
print(summary_task)


# Your Task
You are a rigorous workflow graph extractor and validator.
Your job is to read KNIME workflow XML and produce a clean structural summary of how nodes are connected,
so that the workflow can later be converted into a Galaxy (.ga) workflow.

You must also detect structural and semantic validation errors in the workflow (e.g. missing outputs, invalid connections, type mismatches).
You must not hallucinate any connections, nodes, or ports.
Respond only with valid JSON in the format below — no free text, no comments.

# Input
KNIME Nodes (XML):
```xml
Node ID: Labeling Filter (#8)
<?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="settings.xml">
    <entry key="node_file" type="xstring" value="settings.xml"/>
    <config key="flow_stack"/>
    <config key="internal_node_subset

In [20]:
def build_workflow_examples():
    workflow_examples_text =  """
    Here are some examples of complete KNIME workflows and their corresponding Galaxy workflows:
    """  
    examples = load_translation_examples(yaml_path="data/workflow_translation_table.yml")
    if examples:
        for i, ex in enumerate(examples[:]): 
            workflow_examples_text += f"""

## Example {i + 1}:

KNIME node (XML):

```xml
{ex["KNIME"]}
```

Galaxy step (JSON):
```json
{ex["Galaxy"]}
```

"""
    return workflow_examples_text

In [21]:
workflow_examples = build_workflow_examples()
print(workflow_examples)

[[{'description': 'Example 1 resize rotate', 'KNIME': '<?xml version="1.0" encoding="UTF-8"?>\n<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="workflow.knime">\n    <entry key="created_by" type="xstring" value="4.7.7.v202308161346"/>\n    <entry key="created_by_nightly" type="xboolean" value="false"/>\n    <entry key="version" type="xstring" value="4.1.0"/>\n    <entry key="name" type="xstring" isnull="true" value=""/>\n    <config key="authorInformation">\n        <entry key="authored-by" type="xstring" value="massei"/>\n        <entry key="authored-when" type="xstring" value="2025-08-25 13:02:31 +0200"/>\n        <entry key="lastEdited-by" type="xstring" value="massei"/>\n        <entry key="lastEdited-when" type="xstring" value="2025-08-25 13:10:11 +0200"/>\n    </config>\n    <entry key="customDescription" type="xst

In [22]:
full_summary_prompt = f"{node_examples}\n\n{workflow_examples}\n\n{summary_task}"
print(full_summary_prompt)

You are a translator that converts KNIME workflow nodes to Galaxy workflow steps.

Below are examples of how this translation should be done:


## Example 1:

KNIME node (XML):

```xml
<?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="settings.xml">
    <entry key="node_file" type="xstring" value="settings.xml"/>
    <config key="flow_stack"/>
    <config key="internal_node_subsettings">
        <entry key="memory_policy" type="xstring" value="CacheSmallInMemory"/>
    </config>
    <config key="model">
        <config key="check_file_format_Internals">
            <entry key="SettingsModelID" type="xstring" value="SMID_boolean"/>
            <entry key="EnabledStatus" type="xboolean" value="true"/>
        </config>
        <entry key="check_file_format" type="xboolean" value="true"

In [23]:
def prompt_scadsai_llm(message:str, model="openai/gpt-oss-120b"):
    """A prompt helper function that sends a message to ScaDS.AI LLM server at 
    ZIH TU Dresden and returns only the text response.
    """
    
    # convert message in the right format if necessary
    if isinstance(message, str):
        message = [{"role": "user", "content": message}]
    
    # setup connection to the LLM
    client = openai.OpenAI(base_url="https://llm.scads.ai/v1",
                           api_key=os.environ.get('SCADSAI_API_KEY')
    )
    response = client.chat.completions.create(
        model=model,
        messages=message
    )
    
    # extract answer
    return response.choices[0].message.content

In [24]:
summary_answer = prompt_scadsai_llm(message= full_summary_prompt)

InternalServerError: Error code: 500 - {'error': {'message': "litellm.InternalServerError: InternalServerError: OpenAIException - Connection error.. Received Model Group=openai/gpt-oss-120b\nAvailable Model Group Fallbacks=['meta-llama/Llama-3.3-70B-Instruct', 'deepseek-ai/DeepSeek-R1']\nError doing the fallback: litellm.InternalServerError: InternalServerError: OpenAIException - Connection error.. Received Model Group=deepseek-ai/DeepSeek-R1\nAvailable Model Group Fallbacks=['meta-llama/Llama-4-Scout-17B-16E-Instruct', 'meta-llama/Llama-3.3-70B-Instruct', 'openai/gpt-oss-120b']\nError doing the fallback: litellm.InternalServerError: InternalServerError: OpenAIException - Connection error.. Received Model Group=openai/gpt-oss-120b\nAvailable Model Group Fallbacks=['meta-llama/Llama-3.3-70B-Instruct', 'deepseek-ai/DeepSeek-R1']\nError doing the fallback: litellm.InternalServerError: InternalServerError: OpenAIException - Connection error.", 'type': None, 'param': None, 'code': '500'}}

In [ ]:
print(summary_answer)

In [ ]:
def build_description_task_prompt():
    
    description_task_prompt = f"""
You will receive a KNIME workflow in JSON format.
For each node (step) in the workflow, write a 3 to 5 sentences description 
of what that node does, using simple technical verbs (e.g. trim, filter, convert, normalize, cluster).

Separate each node description with a semicolon (;).
Do not number the items or add any extra text.


Here is the KNIME workflow:
# Input
KNIME Nodes (XML):
```xml
{knime_nodes_str}
```

The KNIME workflow content (XML):
```xml
{workflow_content}
```

This KNIME graph:
{summary_answer}

Output Requirements:
- Separate each node description with a semicolon (;).
- Do not number the items or add any extra text.
- Use no special characters.


"""
    
    return description_task_prompt

In [ ]:
description_task = build_description_task_prompt()
print(description_task)

In [ ]:
full_description_prompt = f"{node_examples}\n\n{workflow_examples}\n\n{description_task}"

In [ ]:
description = prompt_scadsai_llm(message= full_description_prompt)
print(description)

In [ ]:
steps = [s.strip() for s in description.split(";") if s.strip()]
print(steps)

In [ ]:
for step in steps:
    hits = vector_store.search(step, k=10)

In [ ]:
print(hits)

In [ ]:
with open("data/input_workflows.ga", "r", encoding="utf-8") as f:
    input_tools = json.load(f)

In [ ]:
def build_task_prompt():
    
    task_prompt = f"""
# Your Task
You are a system that translates complete KNIME workflows into Galaxy workflows. Produce a **single, valid Galaxy .ga workflow JSON** that can be imported in Galaxy, representing the entire KNIME workflow below.

# Input
KNIME Nodes (XML):
```xml
{knime_nodes_str}
```

The KNIME workflow content (XML):
```xml
{workflow_content}
```

This KNIME graph:
{summary_answer}

The input node descriptions could be one of the following:
{input_tools}

# Output Requirements
- Respond with the complete Galaxy workflow JSON object ONLY (no markdown fences, no comments, no explanations).
- The JSON must be a valid Galaxy .ga workflow 
- Make sure that it is a valid JSON object.
- For uuid fields, write 00000000-0000-0000-0000-000000000000 as placeholder
- Do not include TODOs or comments in the JSON.
- Do not add anything in there that is not part of the Galaxy workflow JSON format
- Get the tool ids, content ids and versions correct based on the following knowledge base of Galaxy tools:

{hits}

- Use type: "data_input" only for a single dataset that is consumed by inputs expecting a single dataset.
- Use type: "data_collection_input" only when any downstream input expects a collection (e.g., list or list:paired).
- Never connect a data_input as a source if the downstream port expects a collection.
- Never invent an "output" on data_input. If an edge would reference such an output, remove that edge and proceed only with valid edges.


- Return a single JSON object and nothing else.

"""
    
    return task_prompt

In [ ]:
task = build_task_prompt()
print(task)

In [ ]:
full_prompt = f"{node_examples}\n\n{workflow_examples}\n\n{task}"
print(full_prompt)

In [ ]:
answer = prompt_scadsai_llm(message= full_prompt)
print(answer)

In [ ]:
# Parse the answer to a JSON object

try: 
    json_object = json.loads(answer)
    print("Parsed JSON successfully.")
    print(json_object)
except json.JSONDecodeError as e:
    print("Failed to parse JSON:", e)


In [ ]:
if "uuid" in json_object:
    json_object["uuid"] = str(uuid.uuid4())

In [ ]:
# Go through the JSON object recursively searching for the "uuid" key and replace the correspondign value with a new UUID
# Use the uuid module to generate a new UUID
for step in json_object["steps"].values():
    if isinstance(step, dict) and "uuid" in step:
        step["uuid"] = str(uuid.uuid4())

In [ ]:
print(json_object["uuid"])
print([s["uuid"] for s in json_object["steps"].values()])

In [ ]:
# Save the answer to a file
output_file = "data/output_file/knime2galaxy_workflow_output3.ga"
with open(output_file, "w", encoding="utf-8") as f:
      json.dump(json_object, f, indent=2, ensure_ascii=False)

print(f"Galaxy workflow saved to {output_file}")